<a href="https://colab.research.google.com/github/nriveramell-ops/23f-desclasificado/blob/Nathaly_Rivera/Proyecto_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Carga de librerias

In [8]:
# Ejecuta esto en una celda si no tienes el modelo
!python -m spacy download es_core_news_sm

import spacy
import re
import os
from spacy.lang.es.stop_words import STOP_WORDS
from google.colab import drive
drive.mount('/content/drive')

# Cargamos el modelo de spaCy para español
nlp = spacy.load("es_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 66.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Carga de archivos

In [11]:
import os

drive_root_path = '/content/drive/MyDrive'

# List contents of the Google Drive root directory
print(f"Contents of '{drive_root_path}':")
for item in os.listdir(drive_root_path):
    print(item)

Contents of '/content/drive/MyDrive':
Colab Notebooks
Machine Learning
Deep Learning
Claúsula de confidencialidad - Alumno TFM.gdoc
Copy_of_NoteBook_JERAR2526_Caso_Tarea_Nathaly.ipynb
Clase 2025_2026 UNAV FE_master
Introduccion TFM.gdoc


In [12]:
import os

# 1. Define la ruta de la carpeta donde están tus .txt
# Si el script está en la misma carpeta que los archivos, usa "."
ruta_carpeta = "/content/drive/MyDrive/Machine Learning"

textos_brutos = []
nombres_archivos = []

# 2. Recorrer la carpeta y leer solo los archivos de texto
for nombre_archivo in os.listdir(ruta_carpeta):
    if nombre_archivo.endswith(".txt"):
        ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)

        # Usamos encoding='utf-8' y errors='ignore' por si hay caracteres extraños del escáner
        with open(ruta_completa, 'r', encoding='utf-8', errors='ignore') as f:
            contenido = f.read()
            textos_brutos.append(contenido)
            nombres_archivos.append(nombre_archivo)

print(f"Se han cargado {len(textos_brutos)} archivos correctamente.")

Se han cargado 157 archivos correctamente.


## Stopwords

In [17]:
# Lista de palabras que queremos ignorar porque no aportan contenido temático al 23-F
custom_stop_words = [
    "página", "fecha", "madrid", "señor", "don", "excelentísimo", "asunto",
    "oficio", "número", "ref", "microfilmación", "folio", "mencionados",
    "anexo", "firmado", "destinado", "perteneciente", "dios", "guarde", "años"
]

# Añadirlas a las stop_words de spaCy
for word in custom_stop_words:
    STOP_WORDS.add(word)


# Ejemplo de resultado:
print(f"custom_stop_words: {len(custom_stop_words)}")

custom_stop_words: 20


## Limpieza y lematizacion

In [23]:
def limpiar_texto(texto):
    # 1. Convertir a minúsculas
    texto = texto.lower()

    # 2. Eliminar números y caracteres especiales (ruido de OCR)
    # Mantenemos solo letras y espacios
    texto = re.sub(r'[^a-záéíóúñ\s]', '', texto)

    # 3. Eliminar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()

    # 4. Procesar con spaCy para Lematización y Stopwords
    doc = nlp(texto)

    # Extraemos el lema si:
    # - No es una stopword
    # - La palabra tiene más de 2 caracteres (evita ruidos de escáner)
    # - Es un sustantivo, verbo o adjetivo (opcional, pero recomendado para Topic Modeling)
    tokens_limpios = [
        token.lemma_ for token in doc
        if token.text not in STOP_WORDS
        and len(token.text) > 2
        and token.pos_ in ["NOUN", "VERB", "ADJ", "PROPN"]
    ]
    return tokens_limpios


In [16]:
path_carpeta = "/content/drive/MyDrive/Machine Learning"  # Cambia esto por tu ruta real
documentos_procesados = []
nombres_archivos = []

# Iteramos por cada archivo subido
for archivo in os.listdir(path_carpeta):
    if archivo.endswith(".txt"):
        with open(os.path.join(path_carpeta, archivo), 'r', encoding='utf-8', errors='ignore') as f:
            contenido = f.read()
            # Aplicamos la limpieza
            tokens = limpiar_texto(contenido)
            documentos_procesados.append(tokens)
            nombres_archivos.append(archivo)

# Ejemplo de resultado:
print(f"Documentos procesados: {len(documentos_procesados)}")
print(f"Primeros tokens del primer documento: {documentos_procesados[0][:10]}")

Documentos procesados: 157
Primeros tokens del primer documento: ['zczc', 'jun', 'dta', 'jun', 'cuartel', 'general', 'jujem', 'gobierno', 'orden', 'general']


## Paso 2

In [30]:
!pip install gensim

In [33]:
# Importamos el módulo corpora de gensim
from gensim import corpora

# Asumimos que del Paso 1 ya tienes la variable 'documentos_procesados'
# que es una lista de listas con las palabras procesadas de cada documento.
# Ejemplo: documentos_procesados = [['militar', 'consejo', 'sentencia'], ['cuartel', 'tensión', 'militar'], ...]

# 1. Crear el Diccionario
# Esto escanea todos los textos y le asigna un ID numérico único a cada palabra distinta.
diccionario = corpora.Dictionary(documentos_procesados)

# Opcional (pero muy recomendado para proyectos de Máster):
# Filtrar palabras que aparecen en muy pocos documentos (menos de 2)
# o en demasiados (en más del 90% de los documentos), ya que no ayudan a diferenciar temas.
# diccionario.filter_extremes(no_below=2, no_above=0.9)

# 2. Crear el Corpus (Bag of Words)
# Para cada documento en tus textos limpios, cuenta la frecuencia de cada ID de palabra.
corpus = [diccionario.doc2bow(texto) for texto in documentos_procesados]

# --- Comprobación de los resultados ---

print(f"Número total de palabras únicas en el diccionario: {len(diccionario)}")
print(f"Número de documentos en el corpus: {len(corpus)}\n")

# Vamos a ver cómo ha quedado el primer documento convertido a números
print(f"Corpus del documento 1 (ID_palabra, Frecuencia):")
print(corpus[0][:10]) # Muestra los primeros 10 elementos del primer documento

# Para entender qué significa cada ID, podemos preguntarle al diccionario:
print("\nTraducción de los primeros 5 IDs del documento 1:")
for id_palabra, frecuencia in corpus[0][:5]:
    palabra_real = diccionario[id_palabra]
    print(f"ID {id_palabra} -> '{palabra_real}' (Aparece {frecuencia} veces)")

Número total de palabras únicas en el diccionario: 14405
Número de documentos en el corpus: 157

Corpus del documento 1 (ID_palabra, Frecuencia):
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 2)]

Traducción de los primeros 5 IDs del documento 1:
ID 0 -> 'acatar' (Aparece 1 veces)
ID 1 -> 'actuacion' (Aparece 1 veces)
ID 2 -> 'actuar' (Aparece 1 veces)
ID 3 -> 'afectar' (Aparece 1 veces)
ID 4 -> 'alvaro' (Aparece 1 veces)


In [34]:
# Importamos el modelo LDA de gensim
from gensim.models import LdaModel

# 1. Definir el número de temas (K)
# Como tienes un dataset pequeño (alrededor de 10-15 documentos del 23-F),
# si le pedimos muchos temas, se va a confundir. Empezaremos con 4.
numero_temas = 4

print(f"Entrenando modelo LDA para descubrir {numero_temas} temas...")

# 2. Entrenamiento del Modelo
lda_model = LdaModel(
    corpus=corpus,                 # El Bag of Words del Paso 2
    id2word=diccionario,           # El diccionario que mapea IDs a palabras del Paso 2
    num_topics=numero_temas,       # El número de tópicos que queremos extraer
    random_state=42,               # MUY IMPORTANTE: Fija una semilla para que tus resultados sean reproducibles (que el Tema 1 siempre sea el mismo si vuelves a ejecutar)
    passes=20,                     # Número de veces que el algoritmo leerá todo el corpus (como las 'epochs' en Deep Learning)
    chunksize=10,                  # Cuántos documentos procesa a la vez
    alpha='auto',                  # Ajusta automáticamente la distribución de temas por documento
    eta='auto'                     # Ajusta automáticamente la distribución de palabras por tema
)

print("¡Modelo entrenado con éxito!\n")

# 3. Mostrar los resultados (Las palabras más importantes de cada tema)
print("Los tópicos descubiertos son los siguientes:\n")

# Le pedimos que nos muestre los temas y las 6 palabras más representativas de cada uno
for id_tema, palabras_clave in lda_model.print_topics(num_topics=numero_temas, num_words=6):
    print(f"Tópico {id_tema}: {palabras_clave}\n")

Entrenando modelo LDA para descubrir 4 temas...
¡Modelo entrenado con éxito!

Los tópicos descubiertos son los siguientes:

Tópico 0: 0.024*"general" + 0.013*"coronel" + 0.011*"ver" + 0.011*"tejero" + 0.011*"guardia" + 0.009*"congreso"

Tópico 1: 0.012*"civil" + 0.011*"militar" + 0.008*"jefe" + 0.008*"general" + 0.007*"gobierno" + 0.006*"fuerza"

Tópico 2: 0.025*"llamar" + 0.023*"decir" + 0.021*"hablar" + 0.019*"pasar" + 0.017*"dejar" + 0.013*"antonio"

Tópico 3: 0.015*"the" + 0.006*"subg" + 0.005*"and" + 0.003*"comando" + 0.003*"reg" + 0.003*"that"



In [37]:
!pip install pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.7 MB/s eta 0:00:00


In [38]:
# Importamos la librería de visualización
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

print("Generando la visualización interactiva. Esto puede tardar unos segundos...")

# 1. Habilitamos la visualización para Jupyter Notebook / Colab
pyLDAvis.enable_notebook()

# 2. Preparamos los datos del modelo para pyLDAvis
# Le pasamos el modelo entrenado, el corpus (bag of words) y el diccionario
visualizacion = gensimvis.prepare(lda_model, corpus, diccionario, sort_topics=False)

# 3. Guardar en un archivo HTML (¡Punto extra para tu entrega final!)
# Así podrás abrirlo en cualquier navegador sin tener que ejecutar Python
pyLDAvis.save_html(visualizacion, 'lda_23F_topics.html')
print("Visualización guardada como 'lda_23F_topics.html'.")

# 4. Mostrar el gráfico directamente en el notebook
visualizacion

Generando la visualización interactiva. Esto puede tardar unos segundos...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Visualización guardada como 'lda_23F_topics.html'.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.187034 -0.196524       1        1  50.164034
1     -0.164795  0.091792       2        1  31.954871
2      0.308734 -0.098416       3        1  15.152150
3      0.043095  0.203147       4        1   2.728945, topic_info=            Term        Freq       Total Category  logprob  loglift
746       llamar  860.000000  860.000000  Default  30.0000  30.0000
469        decir  520.000000  520.000000  Default  29.0000  29.0000
1454      hablar  662.000000  662.000000  Default  28.0000  28.0000
874        pasar  458.000000  458.000000  Default  27.0000  27.0000
476        dejar  471.000000  471.000000  Default  26.0000  26.0000
...          ...         ...         ...      ...      ...      ...
1281      carlos    3.824392   52.384181   Topic4  -6.7630   0.9840
1763  exteriores    3.546220   31.688847   Topic4  -6.8385   1.4112
557       españa    4.016393  287.685369   Topic4  -6.7140  -0.6702
721         juan    3.763079  222.549020   Topic4  -6.7792  -0.4787
1749     asuntos    3.186681   25.731326   Topic4  -6.9454   1.5125

[246 rows x 6 columns], token_table=       Topic      Freq     Term
term                           
1216       1  0.999361  abogado
8246       1  0.047960   abrazo
8246       3  0.949603   abrazo
234        1  0.077945   acción
234        2  0.917347   acción
...      ...       ...      ...
11502      4  0.335137    yerro
2402       4  0.990779      you
172        1  0.992642    órden
172        2  0.005874    órden
1139       1  0.994172    único

[308 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 2, 3, 4])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
